# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id and name
record_sets = list(dataset.record_sets)

print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs.id} | name: {rs.name} | description: {rs.description}")

# List fields for each record set (by @id)
for rs in record_sets:
    print(f"\nFields for Record Set @id={rs.id}:")
    for field in rs.fields:
        print(f"  Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Streams records as a generator, so we convert to list then DataFrame
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} ({len(df)} rows, {len(df.columns)} columns)")
    else:
        print(f"No records found for record set {record_set_id}.")

# For demonstration, pick the first non-empty record set
main_record_set_id = None
for rid in record_set_ids:
    if rid in dataframes and not dataframes[rid].empty:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f"\nColumns in DataFrame ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set contains data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Only run if there's a DataFrame to analyze
import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try to automatically pick a numeric field
    numeric_field = None
    for c in df.columns:
        if np.issubdtype(df[c].dtype, np.number):
            numeric_field = c
            break
    if numeric_field is not None:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
        display(filtered_df.head())

        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Try grouping on a likely categorical column
        group_field = None
        for c in df.columns:
            if c != numeric_field and (df[c].dtype == object or df[c].dtype.name == 'category'):
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped data by {group_field}, showing mean {numeric_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found in record set for EDA.")
else:
    print("No non-empty record set DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id is not None and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None:
        # Boxplot by group
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()
else:
    print('Not enough numeric data for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load, inspect, and perform preliminary exploration on the mass spectrometry dataset using the `mlcroissant` library. Further, we:
- Programmatically accessed available record sets, their fields, and types (all referenced by `@id`),
- Loaded tabular data for further processing and visualized key numeric columns,
- Applied exploratory data analysis such as filtering, normalization, and grouping by categorical variables.

You can repeat this workflow for additional record sets or fields by referencing their `@id` in new cells. Continue your scientific analysis building on this reproducible foundation.